In [ ]:
# (Drive mount removed — notebook reads CSVs from the current directory.)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import datetime
from statsmodels.tsa.stattools import coint
from itertools import combinations

In [ ]:
Data=pd.read_csv("Test_Dataset_No_Sentiment.csv",header=None)

raw=Data.copy()
# Row 0 = tickers, row 1 = price fields
tickers = raw.iloc[0]
fields = raw.iloc[1]

# Actual data starts at row 3, skipping the 'Date' header row which was read as data
df = raw.iloc[3:].copy()

# First column is Date
df = df.rename(columns={0: "Date"})
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")

# Build MultiIndex columns: (ticker, field)
new_cols = []
for ticker, field in zip(tickers[1:], fields[1:]):
    new_cols.append((ticker, field.lower()))

df.columns = pd.MultiIndex.from_tuples(new_cols, names=["ticker", "field"])

# Convert values to numeric
df = df.apply(pd.to_numeric)
df.head()

In [ ]:

def compute_metrics(strategy_returns, periods_per_year=252):
    r = strategy_returns.dropna()

    if len(r) == 0:
        return {
            "Cumulative Return": np.nan,
            "Annualized Volatility": np.nan,
            "Sharpe Ratio": np.nan,
            "Max Drawdown": np.nan,
            "Win Rate": np.nan,
        }

    cumulative_curve = (1 + r).cumprod()

    cumulative_return = cumulative_curve.iloc[-1] - 1

    annualized_volatility = r.std() * np.sqrt(periods_per_year)

    if r.std() == 0:
        sharpe = np.nan
    else:
        sharpe = (r.mean() / r.std()) * np.sqrt(periods_per_year)

    peak = cumulative_curve.cummax()
    drawdown = (cumulative_curve - peak) / peak
    max_drawdown = drawdown.min()

    win_rate = (r > 0).mean()

    return {
        "Cumulative Return": cumulative_return,
        "Annualized Volatility": annualized_volatility,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_drawdown,
        "Win Rate": win_rate,
    }

In [ ]:
tickers = [
    "XOM",
    "CVX",
    "COP",
    "OXY",
   # "PXD":  "Pioneer Natural Resources",
    "SLB",
    "HAL",
    "EOG",
    "DVN",
    "MPC",

    # stocks added by Ryan

    #"PLC": "Principal U.S. Large-Cap Multi-Factor ETF",
    "FANG",
    "PSX",
    "VLO",
    "CTRA",
    "APA"
]

## Single Stock Evaluation

### Momentum (20 day lookback)

In [ ]:

def time_series_momentum_strategy(Data_Frame, stock_name,start_date, end_date,price_col="close", lookback=20, threshold=0.0, allow_short=True):
    # Filter for the specific stock
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    out["return"] = out[price_col].pct_change() # return for each day (rate of change)
    out["momentum"] = out[price_col].pct_change(lookback) # return for 20 days lookback

    if allow_short:
        out["signal"] = np.select(
            [out["momentum"] > threshold, out["momentum"] < -threshold],
            [1, -1],
            default=0
        )
    else:
        out["signal"] = np.where(out["momentum"] > threshold, 1, 0)

    out["position"] = out["signal"].shift(1).fillna(0)

    out["trade"] = out["position"].diff().abs().fillna(0)
    out["strategy_return"] = out["position"] * out["return"]

    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice after features are computed
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_mm = time_series_momentum_strategy(
    df,
    "APA",
    start_date="2022-01-03",
    end_date="2023-01-03",
    price_col="close",
    lookback=20,
    threshold=0.02,
    allow_short=True
)

print(result_mm['position'].loc["2026-02-01":"2026-04-10"])

Series([], Name: position, dtype: float64)


In [ ]:
metrics = compute_metrics(result_mm["strategy_return"])

for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: nan
Annualized Volatility: nan
Sharpe Ratio: nan
Max Drawdown: nan
Win Rate: nan


## Mean Reversion

In [ ]:
import numpy as np
import pandas as pd

def mean_reversion_strategy(
    Data_Frame,
    stock_name,
    price_col="Close",
    lookback=20,
    entry_z=2.0,
    exit_z=0.5,
    allow_short=True,
    start_date=None,
    end_date=None
):
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    # 1-period asset return
    out["return"] = out[price_col].pct_change()

    # Rolling mean and std = estimate of "normal" price behavior
    out["rolling_mean"] = out[price_col].rolling(window=lookback).mean()
    out["rolling_std"] = out[price_col].rolling(window=lookback).std()

    # Z-score: how far price is from its recent average
    out["z_score"] = (out[price_col] - out["rolling_mean"]) / out["rolling_std"]

    # Start flat
    out["signal"] = 0

    if allow_short:
        # Price too low relative to average → long
        out.loc[out["z_score"] < -entry_z, "signal"] = 1

        # Price too high relative to average → short
        out.loc[out["z_score"] > entry_z, "signal"] = -1
    else:
        # Long-only version
        out.loc[out["z_score"] < -entry_z, "signal"] = 1

    # Exit when price comes back close to average
    out.loc[out["z_score"].abs() < exit_z, "signal"] = 0

    # Carry position forward until new signal or exit
    out["position"] = out["signal"].replace(0, np.nan).ffill().fillna(0)

    # If exit condition is met, force position to 0
    out.loc[out["z_score"].abs() < exit_z, "position"] = 0

    # Shift position to avoid lookahead bias
    out["position"] = out["position"].shift(1).fillna(0)

    # Trade size/change
    out["trade"] = out["position"].diff().abs().fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative performance
    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice after features are computed
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_mr = mean_reversion_strategy(
    df,
    "APA",
    price_col="close",
    lookback=20,
    entry_z=2.0,
    exit_z=0.5,
    allow_short=True,
    start_date="2022-01-03",
    end_date="2023-01-03"
)

result_mr[[
    "close",
    "rolling_mean",
    "z_score",
    "signal",
    "position",
    "strategy_return"
]].tail()

field,close,rolling_mean,z_score,signal,position,strategy_return
Date,,,,,,


In [ ]:
metrics_mr = compute_metrics(result_mr["strategy_return"])

for k, v in metrics_mr.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: nan
Annualized Volatility: nan
Sharpe Ratio: nan
Max Drawdown: nan
Win Rate: nan


### Closing Range Breakout

In [ ]:

def closing_range_breakout(
    Data_Frame,
    stock_name,
    price_col="close",
    lookback=20,
    upper_th=0.8,
    lower_th=0.2,
    allow_short=True,
    start_date=None,
    end_date=None
):
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    # Returns
    out["return"] = out[price_col].pct_change()

    # Rolling high and low
    out["rolling_high"] = out[price_col].rolling(lookback).max()
    out["rolling_low"]  = out[price_col].rolling(lookback).min()

    # Closing range
    out["cr"] = (out[price_col] - out["rolling_low"]) / (
        out["rolling_high"] - out["rolling_low"]
    )

    # Signal
    if allow_short:
        out["signal"] = np.select(
            [out["cr"] > upper_th, out["cr"] < lower_th],
            [1, -1],
            default=0
        )
    else:
        out["signal"] = np.where(out["cr"] > upper_th, 1, 0)

    # Position (avoid lookahead bias)
    out["position"] = out["signal"].shift(1).fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative returns
    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice after computing features
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_crb = closing_range_breakout(
    df,
    "APA",
    price_col="close",
    lookback=20,
    upper_th=0.8,
    lower_th=0.2,
    allow_short=True,
    start_date="2022-01-03",
    end_date="2023-01-03"
)

result_crb[[
    "close",
    "rolling_high",
    "rolling_low",
    "cr",
    "signal",
    "position",
    "strategy_return",
    "cumulative_strategy_return"
]].head(40)


field,close,rolling_high,rolling_low,cr,signal,position,strategy_return,cumulative_strategy_return
Date,,,,,,,,


In [ ]:
metrics_crb = compute_metrics(result_crb["strategy_return"])

for k, v in metrics_crb.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: nan
Annualized Volatility: nan
Sharpe Ratio: nan
Max Drawdown: nan
Win Rate: nan


### Turnaround Tueasday - IBS (Internal Bar Strength)

In [ ]:
def turnaround_tuesday_ibs(
    Data_Frame,
    stock_name,
    ibs_threshold=0.2,
    start_date=None,
    end_date=None
):
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    # Returns
    out["return"] = out["close"].pct_change()

    # IBS
    out["ibs"] = (out["close"] - out["low"]) / (out["high"] - out["low"])

    # Day of week (Monday = 0, Tuesday = 1, ...)
    out["weekday"] = pd.to_datetime(out.index).weekday

    # Initialize position
    out["position"] = 0

    # Entry: Monday IBS signal → position on Tuesday
    entry_signal = (out["weekday"] == 0) & (out["ibs"] < ibs_threshold)

    # Shift to enter on Tuesday
    out.loc[entry_signal.shift(1, fill_value=False), "position"] = 1

    # Exit after 1 day → automatically handled since no carry forward

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative returns
    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_ibs = turnaround_tuesday_ibs(
    df,
    "APA",
    ibs_threshold=0.2,
    start_date="2022-01-03",
    end_date="2023-01-03"
)

result_ibs[result_ibs["position"]!=0]

field,close,high,low,open,volume,return,ibs,weekday,position,strategy_return,cumulative_strategy_return,cumulative_asset_return
Date,,,,,,,,,,,,


In [ ]:
metrics_ibs = compute_metrics(result_ibs["strategy_return"])

for k, v in metrics_ibs.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: nan
Annualized Volatility: nan
Sharpe Ratio: nan
Max Drawdown: nan
Win Rate: nan


### ATR (Average True Value) Breakout

In [ ]:

def atr_breakout(
    Data_Frame,
    stock_name,
    high_col="high",
    low_col="low",
    close_col="close",
    atr_lookback=14,
    multiplier=2.0,
    allow_short=True,
    start_date=None,
    end_date=None
):
    stock_df = Data_Frame[stock_name].copy()

    #start from the first available date
    # on or after start_date, handling non-trading days.
    out = stock_df.loc[start_date:].copy()

    # Returns
    out["return"] = out[close_col].pct_change()

    # True Range
    out["tr"] = np.maximum.reduce([
        out[high_col] - out[low_col],
        (out[high_col] - out[close_col].shift(1)).abs(),
        (out[low_col] - out[close_col].shift(1)).abs()
    ])

    # ATR
    out["atr"] = out["tr"].rolling(atr_lookback).mean()

    # Breakout levels (use previous values to avoid lookahead bias)
    out["upper"] = out[close_col].shift(1) + multiplier * out["atr"].shift(1)
    out["lower"] = out[close_col].shift(1) - multiplier * out["atr"].shift(1)

    # Signal
    if allow_short:
        out["signal"] = np.select(
            [out[close_col] > out["upper"], out[close_col] < out["lower"]],
            [1, -1],
            default=0
        )
    else:
        out["signal"] = np.where(out[close_col] > out["upper"], 1, 0)

    # Position
    out["position"] = out["signal"].shift(1).fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative returns
    out["cumulative_strategy_return"] = (1 + out["strategy_return"].fillna(0)).cumprod()
    out["cumulative_asset_return"] = (1 + out["return"].fillna(0)).cumprod()

    # Slice
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    return out

In [ ]:
result_atr = atr_breakout(
    df,
    "APA",
    high_col="high",
    low_col="low",
    close_col="close",
    atr_lookback=14,
    multiplier=2.0,
    allow_short=True,
    start_date="2022-01-03",
    end_date="2023-01-03"
)


result_atr[["close","tr","atr","upper","lower","signal","position","strategy_return","cumulative_strategy_return", "cumulative_asset_return"]].tail()


field,close,tr,atr,upper,lower,signal,position,strategy_return,cumulative_strategy_return,cumulative_asset_return
Date,,,,,,,,,,


In [ ]:
metrics_atr = compute_metrics(result_atr["strategy_return"])

for k, v in metrics_atr.items():
    print(f"{k}: {v:.4f}")

Cumulative Return: nan
Annualized Volatility: nan
Sharpe Ratio: nan
Max Drawdown: nan
Win Rate: nan


### Pairs Trading

In [ ]:

def pairs_trading(
    Data_Frame,
    tickers,
    close_col="close",
    lookback=60,
    z_entry=2.0,
    z_exit=0.5,
    start_date=None,
    end_date=None
):
    df = Data_Frame.copy()

    # Select only the 'close' price for the specified tickers
    # Use .loc with a MultiIndex slice to get the correct columns
    prices = df.loc[:, (tickers, close_col)].copy()

    # Rename columns to just the ticker names for easier access later
    # The columns are now (ticker, close_col), we want just ticker
    prices.columns = prices.columns.get_level_values(0)

    # Slice by date
    if start_date is not None:
        prices = prices.loc[start_date:]
    if end_date is not None:
        prices = prices.loc[:end_date]

    # Drop tickers with too many missing values
    prices = prices.dropna(axis=1)

    # Need at least 2 stocks
    if prices.shape[1] < 2:
        raise ValueError("Need at least two valid tickers with price data.")

    # --------------------------------------------------
    # 1. Choose best pair using cointegration test
    # --------------------------------------------------

    best_pair = None
    best_pvalue = np.inf

    for stock_a, stock_b in combinations(prices.columns, 2):
        series_a = prices[stock_a].dropna()
        series_b = prices[stock_b].dropna()

        common_index = series_a.index.intersection(series_b.index)
        series_a = series_a.loc[common_index]
        series_b = series_b.loc[common_index]

        if len(series_a) < lookback:
            continue

        score, pvalue, _ = coint(series_a, series_b)

        if pvalue < best_pvalue:
            best_pvalue = pvalue
            best_pair = (stock_a, stock_b)

    if best_pair is None:
        raise ValueError("No valid pair found.")

    stock_a, stock_b = best_pair

    # --------------------------------------------------
    # 2. Run pairs trading on selected pair
    # --------------------------------------------------

    out = prices[[stock_a, stock_b]].dropna().copy()

    out["log_A"] = np.log(out[stock_a])
    out["log_B"] = np.log(out[stock_b])

    # Estimate hedge ratio: log_A = alpha + beta * log_B
    X = sm.add_constant(out["log_B"])
    model = sm.OLS(out["log_A"], X).fit()
    beta = model.params["log_B"]

    # Spread
    out["spread"] = out["log_A"] - beta * out["log_B"]

    # Rolling mean/std of spread
    out["spread_mean"] = out["spread"].rolling(lookback).mean()
    out["spread_std"] = out["spread"].rolling(lookback).std()

    # Z-score
    out["z_score"] = (out["spread"] - out["spread_mean"]) / out["spread_std"]

    # Signal:
    # +1 = long spread = long A, short B
    # -1 = short spread = short A, long B
    out["signal"] = 0
    out.loc[out["z_score"] < -z_entry, "signal"] = 1
    out.loc[out["z_score"] > z_entry, "signal"] = -1

    # Exit when spread comes back near mean
    out.loc[out["z_score"].abs() < z_exit, "signal"] = 0

    # Stateful position
    out["position"] = out["signal"].replace(0, np.nan).ffill().fillna(0)

    # Force exit when z-score is close to 0
    out.loc[out["z_score"].abs() < z_exit, "position"] = 0

    # Shift to avoid lookahead bias
    out["position"] = out["position"].shift(1).fillna(0)

    # Returns
    out["ret_A"] = out[stock_a].pct_change()
    out["ret_B"] = out[stock_b].pct_change()

    # Strategy return
    out["strategy_return"] = out["position"] * (out["ret_A"] - beta * out["ret_B"])

    # Cumulative return
    out["cumulative_strategy_return"] = (
        1 + out["strategy_return"].fillna(0)
    ).cumprod()

    # Useful metadata columns
    out["stock_A"] = stock_a
    out["stock_B"] = stock_b
    out["beta"] = beta
    out["cointegration_pvalue"] = best_pvalue

    return out

### Volatility Expension

In [ ]:
def volatility_expansion(
    Data_Frame,
    stock_name,
    close_col="close",
    vol_lookback=20,
    vol_multiplier=1.5,
    allow_short=True,
    start_date=None,
    end_date=None
):
    out = Data_Frame[stock_name].copy()

    # Slice
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    # Returns
    out["return"] = out[close_col].pct_change()

    # Volatility
    out["vol"] = out["return"].rolling(vol_lookback).std()

    # Previous volatility (baseline)
    out["vol_prev"] = out["vol"].rolling(vol_lookback).mean()

    # Signal
    out["signal"] = 0

    # Volatility expansion condition
    vol_expansion = out["vol"] > vol_multiplier * out["vol_prev"]

    if allow_short:
        out.loc[vol_expansion & (out["return"] > 0), "signal"] = 1
        out.loc[vol_expansion & (out["return"] < 0), "signal"] = -1
    else:
        out.loc[vol_expansion & (out["return"] > 0), "signal"] = 1

    # Position
    out["position"] = out["signal"].shift(1).fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative
    out["cumulative_strategy_return"] = (
        1 + out["strategy_return"].fillna(0)
    ).cumprod()

    out["cumulative_asset_return"] = (
        1 + out["return"].fillna(0)
    ).cumprod()

    return out

### Moving Average Crossover

In [ ]:
def moving_average_crossover(
    Data_Frame,
    stock_name,
    close_col="close",
    short_window=20,
    long_window=50,
    allow_short=True,
    start_date=None,
    end_date=None
):
    out = Data_Frame[stock_name].copy()

    # Slice
    if start_date is not None:
        out = out.loc[start_date:]
    if end_date is not None:
        out = out.loc[:end_date]

    # Returns
    out["return"] = out[close_col].pct_change()

    # Moving averages
    out["ma_short"] = out[close_col].rolling(short_window).mean()
    out["ma_long"] = out[close_col].rolling(long_window).mean()

    # Signal
    if allow_short:
        out["signal"] = np.where(
            out["ma_short"] > out["ma_long"], 1, -1
        )
    else:
        out["signal"] = np.where(
            out["ma_short"] > out["ma_long"], 1, 0
        )

    # Position
    out["position"] = out["signal"].shift(1).fillna(0)

    # Strategy return
    out["strategy_return"] = out["position"] * out["return"]

    # Cumulative
    out["cumulative_strategy_return"] = (
        1 + out["strategy_return"].fillna(0)
    ).cumprod()

    out["cumulative_asset_return"] = (
        1 + out["return"].fillna(0)
    ).cumprod()

    return out

### Portfolio Construction

In [ ]:
from dataclasses import dataclass
from typing import Callable, Dict, Optional

REGIMES = ["trending", "mean_reverting", "high_vol"]


def _prepare_regime_probabilities(regime_probabilities: pd.DataFrame) -> pd.DataFrame:
    probs = regime_probabilities.copy()
    rename_map = {
        "p_trending": "trending",
        "p_meanrev": "mean_reverting",
        "p_mean_reverting": "mean_reverting",
        "p_highvol": "high_vol",
        "p_high_vol": "high_vol",
    }
    probs = probs.rename(columns=rename_map)

    missing = [col for col in REGIMES if col not in probs.columns]
    if missing:
        raise ValueError(f"Missing regime probability columns: {missing}")

    probs = probs[REGIMES].astype(float)
    row_sums = probs.sum(axis=1).replace(0.0, np.nan)
    probs = probs.div(row_sums, axis=0)
    return probs.dropna(how="any")


def sample_regime_labels(
    regime_probabilities: pd.DataFrame,
    seed: Optional[int] = 42,
) -> pd.Series:
    probs = _prepare_regime_probabilities(regime_probabilities)
    rng = np.random.default_rng(seed)
    uniforms = rng.random(len(probs))
    cumulative = probs.cumsum(axis=1).to_numpy()
    labels = np.array(REGIMES)[(uniforms[:, None] <= cumulative).argmax(axis=1)]
    return pd.Series(labels, index=probs.index, name="sampled_regime")


def inverse_volatility_weights(
    positions: pd.DataFrame,
    asset_returns: pd.DataFrame,
    vol_lookback: int = 20,
    min_periods: int = 10,
) -> pd.DataFrame:
    rolling_vol = asset_returns.rolling(
        window=vol_lookback,
        min_periods=min_periods,
    ).std().shift(1)

    inverse_vol = 1.0 / rolling_vol.replace(0.0, np.nan)
    signed_scores = positions * inverse_vol
    denominator = signed_scores.abs().sum(axis=1).replace(0.0, np.nan)

    weights = signed_scores.div(denominator, axis=0).fillna(0.0)
    return weights


def build_single_stock_strategy_portfolio(
    data_frame: pd.DataFrame,
    strategy_name: str,
    strategy_fn: Callable[[pd.DataFrame, str], pd.DataFrame],
    tickers: list,
    vol_lookback: int = 20,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
) -> Dict[str, object]:
    positions = {}
    asset_returns = {}
    stock_level_outputs = {}

    for ticker in tickers:
        try:
            strategy_output = strategy_fn(data_frame, ticker)
        except Exception as exc:
            print(f"[{strategy_name}] skipping {ticker}: {exc}")
            continue

        if start_date is not None:
            strategy_output = strategy_output.loc[start_date:]
        if end_date is not None:
            strategy_output = strategy_output.loc[:end_date]

        required_cols = {"position", "return", "strategy_return"}
        if not required_cols.issubset(strategy_output.columns):
            print(f"[{strategy_name}] skipping {ticker}: missing {required_cols - set(strategy_output.columns)}")
            continue

        positions[ticker] = strategy_output["position"].rename(ticker)
        asset_returns[ticker] = strategy_output["return"].rename(ticker)
        stock_level_outputs[ticker] = strategy_output.copy()

    if not positions:
        raise ValueError(f"No valid stock-level outputs for strategy '{strategy_name}'")

    positions_df = pd.concat(positions.values(), axis=1).sort_index()
    returns_df = pd.concat(asset_returns.values(), axis=1).sort_index()

    common_index = positions_df.index.intersection(returns_df.index)
    positions_df = positions_df.loc[common_index].fillna(0.0)
    returns_df = returns_df.loc[common_index].fillna(0.0)

    weights_df = inverse_volatility_weights(
        positions=positions_df,
        asset_returns=returns_df,
        vol_lookback=vol_lookback,
    )
    portfolio_returns = (weights_df * returns_df).sum(axis=1).rename("portfolio_return")
    cumulative_returns = (1.0 + portfolio_returns.fillna(0.0)).cumprod().rename("cumulative_portfolio_return")

    portfolio_frame = pd.concat([portfolio_returns, cumulative_returns], axis=1)

    return {
        "strategy_name": strategy_name,
        "portfolio": portfolio_frame,
        "weights": weights_df,
        "positions": positions_df,
        "asset_returns": returns_df,
        "stock_outputs": stock_level_outputs,
    }


def build_pairs_trading_portfolio(
    data_frame: pd.DataFrame,
    tickers: list,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
) -> Dict[str, object]:
    pairs_output = pairs_trading(
        data_frame,
        tickers=tickers,
        start_date=start_date,
        end_date=end_date,
    ).copy()

    portfolio_frame = pairs_output[["strategy_return", "cumulative_strategy_return"]].rename(
        columns={
            "strategy_return": "portfolio_return",
            "cumulative_strategy_return": "cumulative_portfolio_return",
        }
    )

    return {
        "strategy_name": "Pairs Trading",
        "portfolio": portfolio_frame,
        "weights": None,
        "positions": pairs_output[["position"]],
        "asset_returns": pairs_output[["ret_A", "ret_B"]],
        "stock_outputs": pairs_output,
    }


ARM_BUILDERS = {
    "Momentum": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Momentum",
        strategy_fn=lambda df_, ticker_: time_series_momentum_strategy(
            df_,
            ticker_,
            start_date=start_date,
            end_date=end_date,
            price_col="close",
            lookback=20,
            threshold=0.0,
            allow_short=True,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Mean Reversion": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Mean Reversion",
        strategy_fn=lambda df_, ticker_: mean_reversion_strategy(
            df_,
            ticker_,
            price_col="close",
            lookback=20,
            entry_z=2.0,
            exit_z=0.5,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Closing Range Breakout": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Closing Range Breakout",
        strategy_fn=lambda df_, ticker_: closing_range_breakout(
            df_,
            ticker_,
            price_col="close",
            lookback=20,
            upper_th=0.8,
            lower_th=0.2,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Turnaround Tuesday IBS": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Turnaround Tuesday IBS",
        strategy_fn=lambda df_, ticker_: turnaround_tuesday_ibs(
            df_,
            ticker_,
            ibs_threshold=0.2,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "ATR Breakout": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="ATR Breakout",
        strategy_fn=lambda df_, ticker_: atr_breakout(
            df_,
            ticker_,
            high_col="high",
            low_col="low",
            close_col="close",
            atr_lookback=14,
            multiplier=2.0,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Volatility Expansion": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Volatility Expansion",
        strategy_fn=lambda df_, ticker_: volatility_expansion(
            df_,
            ticker_,
            close_col="close",
            vol_lookback=20,
            vol_multiplier=1.5,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Moving Average Crossover": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Moving Average Crossover",
        strategy_fn=lambda df_, ticker_: moving_average_crossover(
            df_,
            ticker_,
            close_col="close",
            short_window=20,
            long_window=50,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Pairs Trading": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_pairs_trading_portfolio(
        data_frame=data_frame,
        tickers=tickers,
        start_date=start_date,
        end_date=end_date,
    ),
}


def build_arm_return_panel(
    data_frame: pd.DataFrame,
    tickers: list,
    arm_builders: Optional[Dict[str, Callable]] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    vol_lookback: int = 20,
) -> Dict[str, Dict[str, object]]:
    arm_builders = ARM_BUILDERS if arm_builders is None else arm_builders
    arm_outputs = {}

    for arm_name, builder in arm_builders.items():
        arm_outputs[arm_name] = builder(
            data_frame,
            tickers=tickers,
            start_date=start_date,
            end_date=end_date,
            vol_lookback=vol_lookback,
        )

    return arm_outputs

### EXP3 Bandit Algo

In [ ]:
@dataclass
class EXP3:
    n_arms: int
    gamma: float = 0.07
    seed: Optional[int] = 42

    def __post_init__(self):
        self.weights = np.ones(self.n_arms, dtype=float)
        self.rng = np.random.default_rng(self.seed)

    def get_probabilities(self) -> np.ndarray:
        normalized_weights = self.weights / self.weights.sum()
        exploration = np.full(self.n_arms, 1.0 / self.n_arms)
        return (1.0 - self.gamma) * normalized_weights + self.gamma * exploration

    def sample_arm(self) -> tuple[int, np.ndarray]:
        probabilities = self.get_probabilities()
        chosen_arm = int(self.rng.choice(self.n_arms, p=probabilities))
        return chosen_arm, probabilities

    def update(self, chosen_arm: int, reward: float, probabilities: np.ndarray) -> None:
        reward = float(np.clip(reward, 0.0, 1.0))
        estimated_reward = reward / max(probabilities[chosen_arm], 1e-12)
        growth = np.exp((self.gamma * estimated_reward) / self.n_arms)
        self.weights[chosen_arm] *= growth


def scale_reward_to_unit_interval(
    portfolio_return: float,
    reward_clip: float = 0.05,
) -> float:
    """Map period return to [0, 1]; 0 return -> 0.5 (neutral). Not dollars."""
    clipped_return = float(np.clip(portfolio_return, -reward_clip, reward_clip))
    return (clipped_return + reward_clip) / (2.0 * reward_clip)


def rolling_annualized_sharpe_series(
    daily_returns: pd.Series,
    window: int,
    ann_factor: float = 252.0,
) -> pd.Series:
    """Causal rolling Sharpe: at date t uses only returns strictly before t."""
    past = daily_returns.shift(1)
    min_periods = max(5, min(window, window // 2 or 5))
    mu = past.rolling(window, min_periods=min_periods).mean()
    sig = past.rolling(window, min_periods=min_periods).std()
    sharpe = (mu / sig.replace(0.0, np.nan)) * np.sqrt(ann_factor)
    return sharpe


def composite_bandit_reward(
    realized_return: float,
    rolling_sharpe: float,
    *,
    reward_clip: float = 0.05,
    sharpe_clip: float = 2.0,
    w_return: float = 0.5,
    w_sharpe: float = 0.5,
) -> tuple[float, float, float, float]:
    """Combine scaled return and scaled rolling Sharpe into a single [0, 1] reward.

    Returns (bandit_reward, u_return, u_sharpe, sharpe_used).
    Missing/invalid Sharpe uses neutral 0.5 for the Sharpe component (meh, not penalized).
    """
    u_ret = scale_reward_to_unit_interval(realized_return, reward_clip)
    if rolling_sharpe is None or not np.isfinite(rolling_sharpe):
        u_sharp = 0.5
        sharpe_used = float("nan")
    else:
        sharpe_used = float(rolling_sharpe)
        clipped_s = float(np.clip(sharpe_used, -sharpe_clip, sharpe_clip))
        u_sharp = (clipped_s + sharpe_clip) / (2.0 * sharpe_clip)
    ws = float(w_return) + float(w_sharpe)
    if ws <= 0.0:
        wr, ws_ = 1.0, 0.0
    else:
        wr, ws_ = float(w_return) / ws, float(w_sharpe) / ws
    reward = wr * u_ret + ws_ * u_sharp
    return float(np.clip(reward, 0.0, 1.0)), u_ret, u_sharp, sharpe_used


def stock_weights_for_chosen_arm(
    arm_outputs: Dict[str, Dict[str, object]],
    chosen_arm: str,
    date,
    universe_tickers: list,
) -> pd.Series:
    """Per-stock weights for the selected arm on `date` (same scaling as `weights` in builders).

    Single-stock arms: from inverse-vol weights DataFrame. Pairs arm: dollar-neutral split from
    position and hedge ratio `beta` (approximate), since that arm has no per-ticker weight matrix.
    """
    meta = arm_outputs[chosen_arm]
    wdf = meta.get("weights")
    if wdf is not None:
        if date not in wdf.index:
            return pd.Series(np.nan, index=universe_tickers, dtype=float)
        row = wdf.loc[date]
        out = {}
        for t in universe_tickers:
            out[t] = float(row[t]) if t in row.index else np.nan
        return pd.Series(out)

    stock_out = meta.get("stock_outputs")
    if stock_out is None or date not in stock_out.index:
        return pd.Series(np.nan, index=universe_tickers, dtype=float)
    r = stock_out.loc[date]
    need = ("stock_A", "stock_B", "position", "beta")
    if not all(c in r.index for c in need):
        return pd.Series(np.nan, index=universe_tickers, dtype=float)
    A, B = str(r["stock_A"]), str(r["stock_B"])
    pos = float(r["position"])
    beta = float(r["beta"])
    denom = 1.0 + abs(beta)
    w_a = pos * (1.0 / denom)
    w_b = pos * (-beta / denom)
    out = {t: np.nan for t in universe_tickers}
    out[A] = w_a
    out[B] = w_b
    return pd.Series(out)


def run_regime_specific_exp3(
    data_frame: pd.DataFrame,
    regime_probabilities: Optional[pd.DataFrame] = None,
    regime_labels: Optional[pd.Series] = None,
    tickers: Optional[list] = None,
    arm_builders: Optional[Dict[str, Callable]] = None,
    gamma: float = 0.07,
    vol_lookback: int = 20,
    reward_clip: float = 0.05,
    reward_mode: str = "composite",
    sharpe_window: int = 60,
    sharpe_clip: float = 2.0,
    ann_factor: float = 252.0,
    w_return: float = 0.5,
    w_sharpe: float = 0.5,
    seed: Optional[int] = 42,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
) -> Dict[str, object]:
    if regime_labels is None and regime_probabilities is None:
        raise ValueError("Provide either regime_labels or regime_probabilities.")

    if tickers is None:
        tickers = sorted(data_frame.columns.get_level_values("ticker").unique())

    if regime_labels is None:
        regime_labels = sample_regime_labels(regime_probabilities, seed=seed)

    regime_labels = regime_labels.copy()
    regime_labels.index = pd.to_datetime(regime_labels.index)
    if start_date is not None:
        regime_labels = regime_labels.loc[start_date:]
    if end_date is not None:
        regime_labels = regime_labels.loc[:end_date]

    arm_outputs = build_arm_return_panel(
        data_frame=data_frame,
        tickers=tickers,
        arm_builders=arm_builders,
        start_date=start_date,
        end_date=end_date,
        vol_lookback=vol_lookback,
    )

    arm_names = list(arm_outputs.keys())
    arm_return_panel = pd.concat(
        [arm_outputs[arm]["portfolio"]["portfolio_return"].rename(arm) for arm in arm_names],
        axis=1,
    ).sort_index()

    rolling_sharpe_panel = pd.DataFrame(
        {
            arm: rolling_annualized_sharpe_series(
                arm_return_panel[arm], sharpe_window, ann_factor=ann_factor
            )
            for arm in arm_names
        }
    )

    evaluation_dates = arm_return_panel.index.intersection(regime_labels.index)
    arm_return_panel = arm_return_panel.loc[evaluation_dates].fillna(0.0)
    regime_labels = regime_labels.loc[evaluation_dates]

    bandits = {
        regime: EXP3(
            n_arms=len(arm_names),
            gamma=gamma,
            seed=None if seed is None else seed + idx,
        )
        for idx, regime in enumerate(REGIMES)
    }

    history = []

    for date in evaluation_dates:
        regime = regime_labels.loc[date]
        if regime not in bandits:
            continue

        bandit = bandits[regime]
        chosen_idx, probabilities = bandit.sample_arm()
        chosen_arm = arm_names[chosen_idx]
        realized_return = float(arm_return_panel.loc[date, chosen_arm])
        if reward_mode == "return_only":
            reward = scale_reward_to_unit_interval(
                portfolio_return=realized_return,
                reward_clip=reward_clip,
            )
            u_ret = reward
            u_sharp = float("nan")
            sharpe_used = float("nan")
        elif reward_mode == "composite":
            rs = float(rolling_sharpe_panel.loc[date, chosen_arm])
            reward, u_ret, u_sharp, sharpe_used = composite_bandit_reward(
                realized_return,
                rs,
                reward_clip=reward_clip,
                sharpe_clip=sharpe_clip,
                w_return=w_return,
                w_sharpe=w_sharpe,
            )
        else:
            raise ValueError("reward_mode must be 'composite' or 'return_only'.")

        bandit.update(
            chosen_arm=chosen_idx,
            reward=reward,
            probabilities=probabilities,
        )

        w_series = stock_weights_for_chosen_arm(
            arm_outputs, chosen_arm, date, tickers
        )
        weight_dict = {
            str(k): (float(v) if pd.notna(v) else None) for k, v in w_series.items()
        }

        row = {
            "date": date,
            "regime": regime,
            "chosen_arm": chosen_arm,
            "chosen_arm_return": realized_return,
            "bandit_reward": reward,
            "reward_return_part": u_ret,
            "reward_sharpe_part": u_sharp,
            "rolling_sharpe_ann": sharpe_used,
            "stock_weights": weight_dict,
        }
        for t in tickers:
            row[f"w_{t}"] = w_series.loc[t] if t in w_series.index else np.nan

        for arm_idx, arm_name in enumerate(arm_names):
            row[f"p_{arm_name}"] = probabilities[arm_idx]
            row[f"ret_{arm_name}"] = float(arm_return_panel.loc[date, arm_name])

        history.append(row)

    history_df = pd.DataFrame(history).set_index("date").sort_index()

    regime_summaries = {}
    for regime, bandit in bandits.items():
        final_probabilities = bandit.get_probabilities()
        regime_summaries[regime] = pd.Series(final_probabilities, index=arm_names, name=regime)

    regime_probability_summary = pd.DataFrame(regime_summaries).T

    return {
        "history": history_df,
        "arm_outputs": arm_outputs,
        "arm_return_panel": arm_return_panel,
        "rolling_sharpe_panel": rolling_sharpe_panel,
        "bandits": bandits,
        "final_arm_probabilities_by_regime": regime_probability_summary,
        "regime_labels": regime_labels,
    }


# Example usage:
#
# regime_probabilities = pd.read_csv(
#     "lstm_regime_probabilities.csv",
#     parse_dates=["Date"]
# ).set_index("Date")
#
# bandit_results = run_regime_specific_exp3(
#     data_frame=df,
#     regime_probabilities=regime_probabilities,
#     tickers=tickers,
#     gamma=0.07,
#     vol_lookback=20,
#     reward_clip=0.05,
#     seed=42,
# )
#
# bandit_results["history"].head()
# bandit_results["final_arm_probabilities_by_regime"]


if "regime_probabilities" in globals() or "regime_labels" in globals():
    bandit_results = run_regime_specific_exp3(
        data_frame=df,
        regime_probabilities=globals().get("regime_probabilities"),
        regime_labels=globals().get("regime_labels"),
        tickers=tickers,
        gamma=0.07,
        vol_lookback=20,
        reward_clip=0.05,
        reward_mode="composite",
        sharpe_window=60,
        w_return=0.5,
        w_sharpe=0.5,
        seed=42,
    )

    print("Final EXP3 arm probabilities by regime:")
    display(bandit_results["final_arm_probabilities_by_regime"])

    print("\nBandit history preview:")
    display(bandit_results["history"].head())
else:
    print(
        "Bandit framework loaded. Define either `regime_probabilities` "
        "(with columns p_trending, p_meanrev, p_highvol or equivalent) "
        "or `regime_labels`, then rerun this cell."
    )

### Testing the framework

In [ ]:
import matplotlib.pyplot as plt

# Replace these with your actual strategy function names from the notebook.
# Keep only the ones that already exist in your notebook.
ARM_BUILDERS = {
    "Momentum": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Momentum",
        strategy_fn=lambda df_, ticker_: time_series_momentum_strategy(
            df_,
            ticker_,
            start_date=start_date,
            end_date=end_date,
            price_col="close",
            lookback=20,
            threshold=0.0,
            allow_short=True,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Mean Reversion": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Mean Reversion",
        strategy_fn=lambda df_, ticker_: mean_reversion_strategy(
            df_,
            ticker_,
            price_col="close",
            lookback=20,
            entry_z=2.0,
            exit_z=0.5,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Closing Range Breakout": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Closing Range Breakout",
        strategy_fn=lambda df_, ticker_: closing_range_breakout(
            df_,
            ticker_,
            price_col="close",
            lookback=20,
            upper_th=0.8,
            lower_th=0.2,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Turnaround Tuesday IBS": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Turnaround Tuesday IBS",
        strategy_fn=lambda df_, ticker_: turnaround_tuesday_ibs(
            df_,
            ticker_,
            ibs_threshold=0.2,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "ATR Breakout": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="ATR Breakout",
        strategy_fn=lambda df_, ticker_: atr_breakout(
            df_,
            ticker_,
            high_col="high",
            low_col="low",
            close_col="close",
            atr_lookback=14,
            multiplier=2.0,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Volatility Expansion": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Volatility Expansion",
        strategy_fn=lambda df_, ticker_: volatility_expansion(
            df_,
            ticker_,
            close_col="close",
            vol_lookback=20,
            vol_multiplier=1.5,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Moving Average Crossover": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_single_stock_strategy_portfolio(
        data_frame=data_frame,
        strategy_name="Moving Average Crossover",
        strategy_fn=lambda df_, ticker_: moving_average_crossover(
            df_,
            ticker_,
            close_col="close",
            short_window=20,
            long_window=50,
            allow_short=True,
            start_date=start_date,
            end_date=end_date,
        ),
        tickers=tickers,
        vol_lookback=vol_lookback,
        start_date=start_date,
        end_date=end_date,
    ),
    "Pairs Trading": lambda data_frame, tickers, start_date=None, end_date=None, vol_lookback=20: build_pairs_trading_portfolio(
        data_frame=data_frame,
        tickers=tickers,
        start_date=start_date,
        end_date=end_date,
    ),
}

# Align regime probabilities to price data: DatetimeIndex + intersection with df
regime_prob = pd.read_csv("test_regime_probabilities.csv")
_rp_idx = pd.DatetimeIndex(pd.to_datetime(regime_prob["target_date"]).dt.normalize())
regime_probabilities_demo = pd.DataFrame(
    {
        "p_trending": pd.to_numeric(regime_prob["p_trending"], errors="coerce").values,
        "p_meanrev": pd.to_numeric(regime_prob["p_meanrev"], errors="coerce").values,
        "p_highvol": pd.to_numeric(regime_prob["p_highvol"], errors="coerce").values,
    },
    index=_rp_idx,
).sort_index()
regime_probabilities_demo = regime_probabilities_demo.dropna(how="any")
# One row per trading date (keep last if duplicates)
regime_probabilities_demo = regime_probabilities_demo[~regime_probabilities_demo.index.duplicated(keep="last")]

_df_idx = pd.DatetimeIndex(pd.to_datetime(df.index).normalize())
_common = regime_probabilities_demo.index.intersection(_df_idx)
if len(_common) == 0:
    raise ValueError(
        "No overlapping dates between regime CSV (target_date) and price df.index. "
        f"Regime range [{regime_probabilities_demo.index.min()} .. {regime_probabilities_demo.index.max()}], "
        f"df range [{_df_idx.min()} .. {_df_idx.max()}]."
    )
regime_probabilities_demo = regime_probabilities_demo.loc[_common]
demo_dates = regime_probabilities_demo.index

_dmin, _dmax = demo_dates.min(), demo_dates.max()
print(f"Bandit evaluation on {len(demo_dates)} overlapping days ({_dmin.date()} .. {_dmax.date()}).")

bandit_results_demo = run_regime_specific_exp3(
    data_frame=df.loc[_dmin:_dmax],
    regime_probabilities=regime_probabilities_demo,
    tickers=tickers,
    arm_builders=ARM_BUILDERS,
    gamma=0.07,
    vol_lookback=20,
    reward_clip=0.05,
    seed=42,
    start_date=str(_dmin.date()),
    end_date=str(_dmax.date()),
)

history = bandit_results_demo["history"].copy()
final_probs = bandit_results_demo["final_arm_probabilities_by_regime"].copy()

print("Skipped arms:")
# print(bandit_results_demo["skipped_arms"])

print("\nFinal p(arm | regime):")
display(final_probs)

print("\nBandit history:")
display(history)

# Build available probability columns robustly (works even if arm set changes).
prob_cols = [c for c in history.columns if c.startswith("p_")]

# ---------- Improved Visualizations ----------
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.style.use("default")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) Cumulative chosen-arm return
history["cum_chosen_arm_return"] = history["chosen_arm_return"].cumsum()
axes[0, 0].plot(history.index, history["cum_chosen_arm_return"], linewidth=2)
axes[0, 0].set_title("Cumulative Chosen-Arm Return")
axes[0, 0].set_xlabel("Date")
axes[0, 0].set_ylabel("Cumulative Return")

# 2) Bandit reward over time
axes[0, 1].plot(history.index, history["bandit_reward"], linewidth=1.8)
axes[0, 1].set_title("Bandit Reward Over Time")
axes[0, 1].set_xlabel("Date")
axes[0, 1].set_ylabel("Reward [0, 1]")

# 3) Arm selection frequency
arm_counts = history["chosen_arm"].value_counts().sort_values(ascending=False)
axes[1, 0].bar(arm_counts.index, arm_counts.values)
axes[1, 0].set_title("Chosen Arm Frequency")
axes[1, 0].set_xlabel("Arm")
axes[1, 0].set_ylabel("Count")
axes[1, 0].tick_params(axis="x", rotation=45)

# 4) Mean selection probability by arm
if prob_cols:
    mean_probs = history[prob_cols].mean().sort_values(ascending=False)
    axes[1, 1].bar([c[2:] for c in mean_probs.index], mean_probs.values)
    axes[1, 1].set_title("Average p(arm | active regime)")
    axes[1, 1].set_xlabel("Arm")
    axes[1, 1].set_ylabel("Mean Probability")
    axes[1, 1].tick_params(axis="x", rotation=45)
else:
    axes[1, 1].text(0.5, 0.5, "No probability columns found", ha="center", va="center")
    axes[1, 1].set_title("Average p(arm | active regime)")

plt.tight_layout()
plt.show()

# Optional: visualize probability evolution as a heatmap-style image.
if prob_cols:
    prob_matrix = history[prob_cols].T
    fig, ax = plt.subplots(figsize=(16, 5))
    im = ax.imshow(prob_matrix.values, aspect="auto")
    ax.set_yticks(range(len(prob_matrix.index)))
    ax.set_yticklabels([c[2:] for c in prob_matrix.index])
    step = max(1, len(history.index) // 12)
    xticks = list(range(0, len(history.index), step))
    ax.set_xticks(xticks)
    ax.set_xticklabels([pd.Timestamp(history.index[i]).date() for i in xticks], rotation=45, ha="right")
    ax.set_title("Probability Evolution by Arm")
    ax.set_xlabel("Date")
    ax.set_ylabel("Arm")
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Probability")
    plt.tight_layout()
    plt.show()

print("\nDetailed per-day / per-arm outputs:")
# display(bandit_results_demo["debug"])

for date in bandit_results_demo["history"].index:
    row = bandit_results_demo["history"].loc[date]
    # day_debug = bandit_results_demo["debug"][bandit_results_demo["debug"]["date"] == date]

    print("=" * 120)
    print(f"DATE: {pd.Timestamp(date).date()}")
    print(f"REGIME: {row['regime']}")
    print(f"CHOSEN ARM: {row['chosen_arm']}")
    print(f"CHOSEN ARM RETURN: {row['chosen_arm_return']:.6f}")
    print(f"BANDIT REWARD: {row['bandit_reward']:.6f}")
    sw = row.get("stock_weights") or {}
    sw_fmt = {k: round(v, 6) for k, v in sw.items() if v is not None}
    print(f"STOCK WEIGHTS (chosen arm): {sw_fmt}")

    # Dynamically construct probabilities string from existing columns
    probs_info = []
    arm_names_from_builders = list(ARM_BUILDERS.keys())
    for arm_name in arm_names_from_builders:
        prob_col = f'p_{arm_name}'
        if prob_col in row.index:
            probs_info.append(f"{arm_name}: {row[prob_col]:.4f}")
    print(f"P(arm | regime={row['regime']}): {{{', '.join(probs_info)}}}")

    print("-" * 120)

    # The following loop and subsequent display relied on the 'debug' key which is not returned.
    # for _, arm_row in day_debug.iterrows():
    #     chosen_flag = " <= chosen" if arm_row["chosen_by_exp3"] else ""
    #     print(
    #         f"{arm_row['arm']}: "
    #         f"portfolio_return={arm_row['portfolio_return']:.6f}, "
    #         f"scaled_reward={arm_row['scaled_reward']:.6f}{chosen_flag}"
    #     )
    #     print(f"weights: {arm_row['stock_weights']}")
    #     print()

# chosen_only = bandit_results_demo["debug"][bandit_results_demo["debug"]["chosen_by_exp3"]].copy()
# print("\nChosen-arm outcomes:")
# display(chosen_only[["date", "regime", "arm", "portfolio_return", "scaled_reward", "stock_weights"]])